# Project 2 — Experiment Tracking Mastery

**Objective:** Train a Random Forest and a Gradient Boosting model to predict credit default, and log every run's hyperparameters and metrics to MLflow — so no training run is ever lost or unreproducible.

**Input:** the processed arrays produced by `pipeline.py` (Project 1).
**Output:** an MLflow dashboard comparing 12 training runs across both model types.

In [5]:
import sys
!{sys.executable} -m pip install mlflow
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.14.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached alembic-1.18.5-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-3.1.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached skops-0.14.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached databricks_sdk-0.120.0-py3-none-any.whl.metadata (43 kB)
  Using cached opentelemetry_api-1.43.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_proto-1.43.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_sdk-1.43.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached sqlparse-0.5.5-py3-none-any.w

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.38.0 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 which is incompatible.
C:\Users\jesut\anaconda3\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [13]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

## 1. Load the processed data

This is the exact output of `pipeline.py` — no re-cleaning, no re-encoding. That's the whole point of Project 1: every later project just consumes its output.

In [14]:
X_train = np.load("processed/X_train.npy")
X_test = np.load("processed/X_test.npy")
y_train = np.load("processed/y_train.npy")
y_test = np.load("processed/y_test.npy")

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (26059, 26)
X_test: (6515, 26)


## 2. Set up the MLflow experiment

This creates (or reuses) a named experiment. All runs below will be grouped under it, so the dashboard shows a clean comparison table.

In [15]:
mlflow.set_experiment("credit-risk-experiment-tracking")

2026/07/07 13:28:33 INFO mlflow.tracking.fluent: Experiment with name 'credit-risk-experiment-tracking' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:C:/Users/jesut/Downloads/archive (5)/mlruns/1', creation_time=1783427313047, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783427313047, lifecycle_stage='active', name='credit-risk-experiment-tracking', tags={}, trace_location=None, workspace='default'>

## 3. Define hyperparameter combinations

6 Random Forest configs + 6 Gradient Boosting configs = 12 runs total (comfortably over the required 10).

In [16]:
rf_param_grid = [
    {"n_estimators": 50,  "max_depth": 5},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": 10},
    {"n_estimators": 200, "max_depth": None},
    {"n_estimators": 300, "max_depth": None},
]

gb_param_grid = [
    {"n_estimators": 50,  "learning_rate": 0.1,  "max_depth": 3},
    {"n_estimators": 100, "learning_rate": 0.1,  "max_depth": 3},
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 150, "learning_rate": 0.1,  "max_depth": 5},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 5},
    {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": 5},
]

## 4. Train + log function

Each run logs:
- **Parameters:** model type + hyperparameters used
- **Metrics:** Accuracy, F1-Score, AUC-ROC
- **The model itself:** saved as an MLflow artifact, so any run can be reloaded later without retraining

In [17]:
def train_and_log(model_name, model, params):
    with mlflow.start_run(run_name=f"{model_name}_{params}"):
        mlflow.log_param("model_type", model_name)
        for key, value in params.items():
            mlflow.log_param(key, value)

        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        auc = roc_auc_score(y_test, proba)

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("auc_roc", auc)
        mlflow.sklearn.log_model(model, "model")

        print(f"{model_name:<18} {params} -> acc={acc:.4f}  f1={f1:.4f}  auc={auc:.4f}")

## 5. Run all 12 training runs

In [18]:
for params in rf_param_grid:
    train_and_log("RandomForest", RandomForestClassifier(random_state=42, **params), params)

for params in gb_param_grid:
    train_and_log("GradientBoosting", GradientBoostingClassifier(random_state=42, **params), params)

2026/07/07 13:28:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RandomForest       {'n_estimators': 50, 'max_depth': 5} -> acc=0.9176  f1=0.7731  auc=0.8932


2026/07/07 13:29:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RandomForest       {'n_estimators': 100, 'max_depth': 5} -> acc=0.9150  f1=0.7637  auc=0.8948


2026/07/07 13:29:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

GradientBoosting   {'n_estimators': 150, 'learning_rate': 0.1, 'max_depth': 5} -> acc=0.9332  f1=0.8234  auc=0.9456


2026/07/07 13:32:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


GradientBoosting   {'n_estimators': 150, 'learning_rate': 0.05, 'max_depth': 5} -> acc=0.9318  f1=0.8182  auc=0.9345


2026/07/07 13:33:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


GradientBoosting   {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 5} -> acc=0.9340  f1=0.8259  auc=0.9485


## 6. View the dashboard

Run this in your terminal (not in the notebook), from the same folder:

```
mlflow ui
```

Then open **http://localhost:5000** in your browser. You'll see all 12 runs listed, with sortable columns for accuracy, F1, and AUC-ROC — exactly what Artifact 2 on the checklist asks for. Take a screenshot of that table for your portfolio/README.